# Tiny Graphs - Node2Vec (Tied)


## 1) Tiny Path-Star (deg=4, deg_tree=1, path_len=5)


In [ ]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents) if (candidate / "tiny_graphs_notebooks").is_dir()),
    CURRENT_DIR,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import matplotlib.pyplot as plt

from tiny_graphs_notebooks.notebook_utils.node2vec_notebook_helpers import (
    build_node2vec_section_context,
    train_or_load_node2vec_model,
    evaluate_node2vec_edges,
    print_node2vec_evaluation_report,
    collect_node2vec_embeddings_and_edges,
    load_node2vec_analysis_histories,
)
from tiny_graphs_notebooks.analysis.analysis import (
    build_graph_spectral_state,
    compute_associative_geometric_curves,
    compute_laplacian_coordinates,
    compute_spectral_bias_from_state,
    get_star_branch_layout,
    plot_associative_vs_geometric_curves,
    plot_spectral_bias,
    plot_ordered_heatmaps,
    plot_stylized_embedding_graph,
    reduce_embeddings_for_plot,
    select_embedding_snapshot,
)


In [ ]:
path_star_seed = 7

path_star_train_from_scratch = True
path_star_checkpoint_path = ""
path_star_embedding_history_path = ""
path_star_topk_history_path = ""

path_star_config = {
    "graph_type": "star",
    "star_degree": 4,
    "path_length": 5,

    "embedding_dim": 100,
    "learning_rate": 0.01,
    "num_epochs": 10000,
    "neg_samples_per_pos": 3,
    "use_full_softmax_objective": True,
    "embedding_checkpoint_interval": 25,
    "top_k": 5,
}


In [ ]:
path_star_context = build_node2vec_section_context(path_star_config, seed=path_star_seed)

path_star_resolved_checkpoint_path = train_or_load_node2vec_model(
    path_star_context,
    train_from_scratch=path_star_train_from_scratch,
    checkpoint_path=path_star_checkpoint_path,
)

print("Run name:", path_star_context.run_name)
print("Checkpoint:", path_star_resolved_checkpoint_path)
print("Manifest:", path_star_context.manifest_path)
print("Embedding history pickle:", path_star_context.embedding_history_path)
print("Top-k history pickle:", path_star_context.topk_history_path)


In [ ]:
path_star_eval_metrics = evaluate_node2vec_edges(path_star_context)

print_node2vec_evaluation_report("Tiny Path-Star", path_star_eval_metrics)


In [ ]:
path_star_node_embeddings, path_star_edge_list, path_star_root_node_index = collect_node2vec_embeddings_and_edges(path_star_context)

path_star_embedding_history, path_star_topk_recovery_history = load_node2vec_analysis_histories(
    path_star_context,
    embedding_history_path=path_star_embedding_history_path,
    topk_history_path=path_star_topk_history_path,
)

if not path_star_embedding_history:
    path_star_embedding_history = {0: path_star_node_embeddings}
if not path_star_topk_recovery_history:
    path_star_topk_recovery_history = {0: path_star_eval_metrics['topk_edge_recovery_percent']}

print("Node embedding shape:", path_star_node_embeddings.shape)
print("Number of directed edges:", len(path_star_edge_list))
print("Root node index:", path_star_root_node_index)
print("Embedding history steps:", len(path_star_embedding_history))
print("Top-k history steps:", len(path_star_topk_recovery_history))


In [ ]:
# Reduction config for this block (kept local to reduction stage)
path_star_use_umap = False
path_star_reduction_dim = 3
path_star_umap_n_neighbors = 5
path_star_umap_min_dist = 0.3

path_star_reduced_full, path_star_reduced_embeddings = reduce_embeddings_for_plot(
    path_star_node_embeddings,
    use_umap=path_star_use_umap,
    reduction_dim=path_star_reduction_dim,
    seed=path_star_seed,
    umap_n_neighbors=path_star_umap_n_neighbors,
    umap_min_dist=path_star_umap_min_dist,
)

path_star_reduced_embeddings = path_star_reduced_full[:, :3]
print("Reduced full shape:", path_star_reduced_full.shape)
print("Reduced xyz shape:", path_star_reduced_embeddings.shape)


In [ ]:
# %matplotlib widget
%matplotlib inline

# Raw plotting constants for this block
path_star_raw_elev = 15
path_star_raw_azim = -25
path_star_raw_roll = 0
path_star_raw_axis_permutation = (0, 1, 2)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

_xi, _yi, _zi = path_star_raw_axis_permutation
ax.scatter(
    path_star_reduced_embeddings[:, _xi],
    path_star_reduced_embeddings[:, _yi],
    path_star_reduced_embeddings[:, _zi],
    s=180,
    alpha=0.95,
    edgecolor='black',
    linewidth=0.4,
)

for u, v in path_star_edge_list:
    ax.plot(
        [path_star_reduced_embeddings[u, _xi], path_star_reduced_embeddings[v, _xi]],
        [path_star_reduced_embeddings[u, _yi], path_star_reduced_embeddings[v, _yi]],
        [path_star_reduced_embeddings[u, _zi], path_star_reduced_embeddings[v, _zi]],
        color='k',
        linewidth=0.9,
    )

ax.view_init(elev=float(path_star_raw_elev), azim=float(path_star_raw_azim), roll=float(path_star_raw_roll))
plt.title("")
plt.show()


In [ ]:
# Styled plotting constants for this block
path_star_styled_view = {"elev": path_star_raw_elev, "azim": path_star_raw_azim, "roll": path_star_raw_roll}
path_star_styled_axis_permutation = (0, 1, 2)

plot_stylized_embedding_graph(
    path_star_reduced_embeddings,
    path_star_edge_list,
    title="",
    view=path_star_styled_view,
    root_node_index=path_star_root_node_index,
    axis_permutation=path_star_styled_axis_permutation,
);


In [ ]:
# Line-plot constants for this block
path_star_line_plot_title = ""

path_star_curve_steps, path_star_curve_associative, path_star_curve_geometric = compute_associative_geometric_curves(
    path_star_embedding_history,
    path_star_topk_recovery_history,
    path_star_edge_list,
)

plot_associative_vs_geometric_curves(
    steps=path_star_curve_steps,
    associative_scores=path_star_curve_associative,
    geometric_scores=path_star_curve_geometric,
    title=path_star_line_plot_title,
);


Note setting `path_star_heatmap_epoch` to one of the highest geometric_level epochs.


In [ ]:
# Heatmap constants for this block
path_star_heatmap_graph_type = path_star_context.graph_type
path_star_heatmap_epoch = -1
path_star_heatmap_cmap_name = "plasma"
path_star_heatmap_wspace = 0.5

path_star_heatmap_embeddings, path_star_heatmap_resolved_epoch = select_embedding_snapshot(
    embedding_history=path_star_embedding_history,
    fallback_embeddings=path_star_node_embeddings,
    requested_step=path_star_heatmap_epoch,
)
print("Path-star heatmap epoch:", path_star_heatmap_resolved_epoch)
path_star_branch_layout = get_star_branch_layout(
    edge_list=path_star_edge_list,
    node_count=path_star_heatmap_embeddings.shape[0],
    root_node_index=path_star_root_node_index,
)
print("Path-star branch layout:")
for branch_row in path_star_branch_layout:
    print(" ".join(str(node_id) for node_id in branch_row))
if path_star_branch_layout:
    path_star_heatmap_order = [path_star_branch_layout[0][0]]
    for branch_row in path_star_branch_layout:
        path_star_heatmap_order.extend(branch_row[1:])
else:
    path_star_heatmap_order = list(range(path_star_heatmap_embeddings.shape[0]))
print("Path-star heatmap order:", path_star_heatmap_order)

plot_ordered_heatmaps(
    embeddings=path_star_heatmap_embeddings,
    edge_list=path_star_edge_list,
    graph_type=path_star_heatmap_graph_type,
    root_node_index=path_star_root_node_index,
    custom_order=path_star_heatmap_order,
    cmap_name=path_star_heatmap_cmap_name,
    wspace=path_star_heatmap_wspace,
);


### Laplacian Geometry Plot


In [ ]:
path_star_styled_view = {"elev": 15, "azim": -45, "roll": 0}

# Laplacian geometry constants for this block
path_star_laplacian_axis_indices = (-2, -3, -4)
path_star_laplacian_state = build_graph_spectral_state(
    edge_list=path_star_edge_list,
    node_count=path_star_node_embeddings.shape[0],
)
path_star_laplacian_coords = compute_laplacian_coordinates(
    spectral_state=path_star_laplacian_state,
    axis_indices=path_star_laplacian_axis_indices,
)

plot_stylized_embedding_graph(
    path_star_laplacian_coords,
    path_star_edge_list,
    title="",
    view=path_star_styled_view,
    root_node_index=path_star_root_node_index,
    axis_permutation=path_star_styled_axis_permutation,
);


### Skewed Low-Rank Spectral Bias


In [ ]:
# Spectral-bias constants for this block
path_star_spectral_drop_top_eigenvector = True
path_star_spectral_reorder_prefix = (0, 1, 2)
path_star_spectral_cutoff = None # 8
path_star_spectral_figsize = (15.0, 6.0)
path_star_spectral_legend_anchor = (0.0, 1.35)

path_star_norm_eigenvalues, path_star_norm_projections = compute_spectral_bias_from_state(
    embeddings=path_star_node_embeddings,
    spectral_state=path_star_laplacian_state,
    drop_top_eigenvector=path_star_spectral_drop_top_eigenvector,
    reorder_prefix=path_star_spectral_reorder_prefix,
)

print("Path-Star normalized eigenvalues:\n", path_star_norm_eigenvalues.round(3))

plot_spectral_bias(
    norm_eigenvalues=path_star_norm_eigenvalues,
    norm_projections=path_star_norm_projections,
    title="",
    cutoff=path_star_spectral_cutoff,
    figsize=path_star_spectral_figsize,
    ylabel_fontsize=30,
    xlabel_fontsize=30,
    tick_fontsize=28,
    legend_fontsize=20,
    legend_anchor=path_star_spectral_legend_anchor,
);

## 2) Tiny Grid (rows=4, cols=4)


In [ ]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents) if (candidate / "tiny_graphs_notebooks").is_dir()),
    CURRENT_DIR,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import matplotlib.pyplot as plt

from tiny_graphs_notebooks.notebook_utils.node2vec_notebook_helpers import (
    build_node2vec_section_context,
    train_or_load_node2vec_model,
    evaluate_node2vec_edges,
    print_node2vec_evaluation_report,
    collect_node2vec_embeddings_and_edges,
    load_node2vec_analysis_histories,
)
from tiny_graphs_notebooks.analysis.analysis import (
    build_graph_spectral_state,
    compute_associative_geometric_curves,
    compute_laplacian_coordinates,
    compute_spectral_bias_from_state,
    get_star_branch_layout,
    plot_associative_vs_geometric_curves,
    plot_spectral_bias,
    plot_ordered_heatmaps,
    plot_stylized_embedding_graph,
    reduce_embeddings_for_plot,
    select_embedding_snapshot,
)


In [ ]:
grid_seed = 7

grid_train_from_scratch = True
grid_checkpoint_path = ""
grid_embedding_history_path = ""
grid_topk_history_path = ""

grid_config = {
    "graph_type": "grid",
    "grid_rows": 4,
    "grid_cols": 4,
    "path_length": 5,

    "embedding_dim": 100,
    "learning_rate": 0.01,
    "num_epochs": 10000,
    "neg_samples_per_pos": 3,
    "use_full_softmax_objective": True,
    "embedding_checkpoint_interval": 25,
    "top_k": 5,
}


In [ ]:
grid_context = build_node2vec_section_context(grid_config, seed=grid_seed)

grid_resolved_checkpoint_path = train_or_load_node2vec_model(
    grid_context,
    train_from_scratch=grid_train_from_scratch,
    checkpoint_path=grid_checkpoint_path,
)

print("Run name:", grid_context.run_name)
print("Checkpoint:", grid_resolved_checkpoint_path)
print("Manifest:", grid_context.manifest_path)
print("Embedding history pickle:", grid_context.embedding_history_path)
print("Top-k history pickle:", grid_context.topk_history_path)


In [ ]:
grid_eval_metrics = evaluate_node2vec_edges(grid_context)

print_node2vec_evaluation_report("Tiny Grid", grid_eval_metrics)


In [ ]:
grid_node_embeddings, grid_edge_list, grid_root_node_index = collect_node2vec_embeddings_and_edges(grid_context)

grid_embedding_history, grid_topk_recovery_history = load_node2vec_analysis_histories(
    grid_context,
    embedding_history_path=grid_embedding_history_path,
    topk_history_path=grid_topk_history_path,
)

if not grid_embedding_history:
    grid_embedding_history = {0: grid_node_embeddings}
if not grid_topk_recovery_history:
    grid_topk_recovery_history = {0: grid_eval_metrics['topk_edge_recovery_percent']}

print("Node embedding shape:", grid_node_embeddings.shape)
print("Number of directed edges:", len(grid_edge_list))
print("Root node index:", grid_root_node_index)
print("Embedding history steps:", len(grid_embedding_history))
print("Top-k history steps:", len(grid_topk_recovery_history))


In [ ]:
# Reduction config for this block (kept local to reduction stage)
grid_use_umap = False
grid_reduction_dim = 3
grid_umap_n_neighbors = 4
grid_umap_min_dist = 0.3

grid_reduced_full, grid_reduced_embeddings = reduce_embeddings_for_plot(
    grid_node_embeddings,
    use_umap=grid_use_umap,
    reduction_dim=grid_reduction_dim,
    seed=grid_seed,
    umap_n_neighbors=grid_umap_n_neighbors,
    umap_min_dist=grid_umap_min_dist,
)

grid_reduced_embeddings = grid_reduced_full[:, :3]
print("Reduced full shape:", grid_reduced_full.shape)
print("Reduced xyz shape:", grid_reduced_embeddings.shape)


In [ ]:
%matplotlib inline
# %matplotlib widget

# Raw plotting constants for this block
grid_raw_elev = 90
grid_raw_azim = 90
grid_raw_roll = 90
grid_raw_axis_permutation = (0, 1, 2)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

_xi, _yi, _zi = grid_raw_axis_permutation
ax.scatter(
    grid_reduced_embeddings[:, _xi],
    grid_reduced_embeddings[:, _yi],
    grid_reduced_embeddings[:, _zi],
    s=180,
    alpha=0.95,
    edgecolor='black',
    linewidth=0.4,
)

for u, v in grid_edge_list:
    ax.plot(
        [grid_reduced_embeddings[u, _xi], grid_reduced_embeddings[v, _xi]],
        [grid_reduced_embeddings[u, _yi], grid_reduced_embeddings[v, _yi]],
        [grid_reduced_embeddings[u, _zi], grid_reduced_embeddings[v, _zi]],
        color='k',
        linewidth=0.9,
    )

ax.view_init(elev=float(grid_raw_elev), azim=float(grid_raw_azim), roll=float(grid_raw_roll))
plt.title("")
plt.show()


In [ ]:
# Styled plotting constants for this block
grid_styled_view = {"elev": grid_raw_elev, "azim": grid_raw_azim, "roll": grid_raw_roll}
grid_styled_axis_permutation = (0, 1, 2)

plot_stylized_embedding_graph(
    grid_reduced_embeddings,
    grid_edge_list,
    title="",
    view=grid_styled_view,
    root_node_index=grid_root_node_index,
    axis_permutation=grid_styled_axis_permutation,
);


In [ ]:
# Line-plot constants for this block
grid_line_plot_title = ""

grid_curve_steps, grid_curve_associative, grid_curve_geometric = compute_associative_geometric_curves(
    grid_embedding_history,
    grid_topk_recovery_history,
    grid_edge_list,
)

plot_associative_vs_geometric_curves(
    steps=grid_curve_steps,
    associative_scores=grid_curve_associative,
    geometric_scores=grid_curve_geometric,
    title=grid_line_plot_title,
);


Note setting `grid_heatmap_epoch` to one of the highest geometric_level epochs.


In [ ]:
# Heatmap constants for this block
grid_heatmap_graph_type = grid_context.graph_type
grid_heatmap_epoch = -1
grid_heatmap_cmap_name = "plasma"
grid_heatmap_wspace = 0.5

grid_heatmap_embeddings, grid_heatmap_resolved_epoch = select_embedding_snapshot(
    embedding_history=grid_embedding_history,
    fallback_embeddings=grid_node_embeddings,
    requested_step=grid_heatmap_epoch,
)
print("Grid heatmap epoch:", grid_heatmap_resolved_epoch)
grid_heatmap_order = list(range(grid_heatmap_embeddings.shape[0]))
grid_heatmap_row_width = int(grid_context.config.get("grid_cols", 4))
print("Grid heatmap order rows:")
for row_start in range(0, len(grid_heatmap_order), grid_heatmap_row_width):
    row = grid_heatmap_order[row_start:row_start + grid_heatmap_row_width]
    print(" ".join(str(node_id) for node_id in row))

plot_ordered_heatmaps(
    embeddings=grid_heatmap_embeddings,
    edge_list=grid_edge_list,
    graph_type=grid_heatmap_graph_type,
    root_node_index=grid_root_node_index,
    custom_order=grid_heatmap_order,
    cmap_name=grid_heatmap_cmap_name,
    wspace=grid_heatmap_wspace,
);


### Laplacian Geometry Plot


In [ ]:
grid_styled_view = {"elev": 90, "azim": 90, "roll": 90}

# Laplacian geometry constants for this block
grid_laplacian_axis_indices = (-2, -3, -4)
grid_laplacian_state = build_graph_spectral_state(
    edge_list=grid_edge_list,
    node_count=grid_node_embeddings.shape[0],
)
grid_laplacian_coords = compute_laplacian_coordinates(
    spectral_state=grid_laplacian_state,
    axis_indices=grid_laplacian_axis_indices,
)

plot_stylized_embedding_graph(
    grid_laplacian_coords,
    grid_edge_list,
    title="",
    view=grid_styled_view,
    root_node_index=grid_root_node_index,
    axis_permutation=grid_styled_axis_permutation,
);

### Skewed Low-Rank Spectral Bias


In [ ]:
# Spectral-bias constants for this block
grid_spectral_drop_top_eigenvector = True
grid_spectral_reorder_prefix = None
grid_spectral_cutoff = None
grid_spectral_figsize = (15.0, 6.0)
grid_spectral_legend_anchor = (0.0, 1.3)

grid_norm_eigenvalues, grid_norm_projections = compute_spectral_bias_from_state(
    embeddings=grid_node_embeddings,
    spectral_state=grid_laplacian_state,
    drop_top_eigenvector=grid_spectral_drop_top_eigenvector,
    reorder_prefix=grid_spectral_reorder_prefix,
)

print("Grid normalized eigenvalues:\n", grid_norm_eigenvalues.round(3))

plot_spectral_bias(
    norm_eigenvalues=grid_norm_eigenvalues,
    norm_projections=grid_norm_projections,
    title="",
    cutoff=grid_spectral_cutoff,
    figsize=grid_spectral_figsize,
    ylabel_fontsize=28,
    xlabel_fontsize=28,
    tick_fontsize=20,
    legend_fontsize=20,
    legend_anchor=grid_spectral_legend_anchor,
);

## 3) Tiny Cycle (N=15)


In [ ]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents) if (candidate / "tiny_graphs_notebooks").is_dir()),
    CURRENT_DIR,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import matplotlib.pyplot as plt

from tiny_graphs_notebooks.notebook_utils.node2vec_notebook_helpers import (
    build_node2vec_section_context,
    train_or_load_node2vec_model,
    evaluate_node2vec_edges,
    print_node2vec_evaluation_report,
    collect_node2vec_embeddings_and_edges,
    load_node2vec_analysis_histories,
)
from tiny_graphs_notebooks.analysis.analysis import (
    build_graph_spectral_state,
    compute_associative_geometric_curves,
    compute_laplacian_coordinates,
    compute_spectral_bias_from_state,
    get_star_branch_layout,
    plot_associative_vs_geometric_curves,
    plot_spectral_bias,
    plot_ordered_heatmaps,
    plot_stylized_embedding_graph,
    reduce_embeddings_for_plot,
    select_embedding_snapshot,
)


In [ ]:
cycle_seed = 7

cycle_train_from_scratch = True
cycle_checkpoint_path = ""
cycle_embedding_history_path = ""
cycle_topk_history_path = ""

cycle_config = {
    "graph_type": "cycle",
    "total_nodes": 15,
    "path_length": 5,

    "embedding_dim": 100,
    "learning_rate": 0.01,
    "num_epochs": 10000,
    "neg_samples_per_pos": 3,
    "use_full_softmax_objective": True,
    "embedding_checkpoint_interval": 25,
    "top_k": 5,
}


In [ ]:
cycle_context = build_node2vec_section_context(cycle_config, seed=cycle_seed)

cycle_resolved_checkpoint_path = train_or_load_node2vec_model(
    cycle_context,
    train_from_scratch=cycle_train_from_scratch,
    checkpoint_path=cycle_checkpoint_path,
)

print("Run name:", cycle_context.run_name)
print("Checkpoint:", cycle_resolved_checkpoint_path)
print("Manifest:", cycle_context.manifest_path)
print("Embedding history pickle:", cycle_context.embedding_history_path)
print("Top-k history pickle:", cycle_context.topk_history_path)


In [ ]:
cycle_eval_metrics = evaluate_node2vec_edges(cycle_context)

print_node2vec_evaluation_report("Tiny Cycle", cycle_eval_metrics)


In [ ]:
cycle_node_embeddings, cycle_edge_list, cycle_root_node_index = collect_node2vec_embeddings_and_edges(cycle_context)

cycle_embedding_history, cycle_topk_recovery_history = load_node2vec_analysis_histories(
    cycle_context,
    embedding_history_path=cycle_embedding_history_path,
    topk_history_path=cycle_topk_history_path,
)

if not cycle_embedding_history:
    cycle_embedding_history = {0: cycle_node_embeddings}
if not cycle_topk_recovery_history:
    cycle_topk_recovery_history = {0: cycle_eval_metrics['topk_edge_recovery_percent']}

print("Node embedding shape:", cycle_node_embeddings.shape)
print("Number of directed edges:", len(cycle_edge_list))
print("Root node index:", cycle_root_node_index)
print("Embedding history steps:", len(cycle_embedding_history))
print("Top-k history steps:", len(cycle_topk_recovery_history))


In [ ]:
# Reduction config for this block (kept local to reduction stage)
cycle_use_umap = False
cycle_reduction_dim = 3
cycle_umap_n_neighbors = 8
cycle_umap_min_dist = 0.3

cycle_reduced_full, cycle_reduced_embeddings = reduce_embeddings_for_plot(
    cycle_node_embeddings,
    use_umap=cycle_use_umap,
    reduction_dim=cycle_reduction_dim,
    seed=cycle_seed,
    umap_n_neighbors=cycle_umap_n_neighbors,
    umap_min_dist=cycle_umap_min_dist,
)

cycle_reduced_embeddings = cycle_reduced_full[:, :3]
print("Reduced full shape:", cycle_reduced_full.shape)
print("Reduced xyz shape:", cycle_reduced_embeddings.shape)


In [ ]:
%matplotlib inline
# %matplotlib widget

# Raw plotting constants for this block
cycle_raw_elev = 90
cycle_raw_azim = 145
cycle_raw_roll = 59
cycle_raw_axis_permutation = (0, 1, 2)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

_xi, _yi, _zi = cycle_raw_axis_permutation
ax.scatter(
    cycle_reduced_embeddings[:, _xi],
    cycle_reduced_embeddings[:, _yi],
    cycle_reduced_embeddings[:, _zi],
    s=180,
    alpha=0.95,
    edgecolor='black',
    linewidth=0.4,
)

for u, v in cycle_edge_list:
    ax.plot(
        [cycle_reduced_embeddings[u, _xi], cycle_reduced_embeddings[v, _xi]],
        [cycle_reduced_embeddings[u, _yi], cycle_reduced_embeddings[v, _yi]],
        [cycle_reduced_embeddings[u, _zi], cycle_reduced_embeddings[v, _zi]],
        color='k',
        linewidth=0.9,
    )

ax.view_init(elev=float(cycle_raw_elev), azim=float(cycle_raw_azim), roll=float(cycle_raw_roll))
plt.title("")
plt.show()


In [ ]:
# Styled plotting constants for this block
cycle_styled_view = {"elev": cycle_raw_elev, "azim": cycle_raw_azim, "roll": cycle_raw_roll}
cycle_styled_axis_permutation = (0, 1, 2)

plot_stylized_embedding_graph(
    cycle_reduced_embeddings,
    cycle_edge_list,
    title="",
    view=cycle_styled_view,
    root_node_index=cycle_root_node_index,
    axis_permutation=cycle_styled_axis_permutation,
);


In [ ]:
# Line-plot constants for this block
cycle_line_plot_title = ""

cycle_curve_steps, cycle_curve_associative, cycle_curve_geometric = compute_associative_geometric_curves(
    cycle_embedding_history,
    cycle_topk_recovery_history,
    cycle_edge_list,
)

plot_associative_vs_geometric_curves(
    steps=cycle_curve_steps,
    associative_scores=cycle_curve_associative,
    geometric_scores=cycle_curve_geometric,
    title=cycle_line_plot_title,
);


Note setting `cycle_heatmap_epoch` to one of the highest geometric_level epochs.


In [ ]:
# Heatmap constants for this block
cycle_heatmap_graph_type = cycle_context.graph_type
cycle_heatmap_epoch = -1
cycle_heatmap_cmap_name = "plasma"
cycle_heatmap_wspace = 0.5

cycle_heatmap_embeddings, cycle_heatmap_resolved_epoch = select_embedding_snapshot(
    embedding_history=cycle_embedding_history,
    fallback_embeddings=cycle_node_embeddings,
    requested_step=cycle_heatmap_epoch,
)
print("Cycle heatmap epoch:", cycle_heatmap_resolved_epoch)
cycle_heatmap_order = list(range(cycle_heatmap_embeddings.shape[0]))
print("Cycle heatmap order:")
print(" ".join(str(node_id) for node_id in cycle_heatmap_order))

plot_ordered_heatmaps(
    embeddings=cycle_heatmap_embeddings,
    edge_list=cycle_edge_list,
    graph_type=cycle_heatmap_graph_type,
    root_node_index=cycle_root_node_index,
    custom_order=cycle_heatmap_order,
    cmap_name=cycle_heatmap_cmap_name,
    wspace=cycle_heatmap_wspace,
);


### Laplacian Geometry Plot


In [ ]:
cycle_styled_view = {"elev": 90, "azim": 145, "roll": 59}

# Laplacian geometry constants for this block
cycle_laplacian_axis_indices = (-2, -3, -4)
cycle_laplacian_state = build_graph_spectral_state(
    edge_list=cycle_edge_list,
    node_count=cycle_node_embeddings.shape[0],
)
cycle_laplacian_coords = compute_laplacian_coordinates(
    spectral_state=cycle_laplacian_state,
    axis_indices=cycle_laplacian_axis_indices,
)

plot_stylized_embedding_graph(
    cycle_laplacian_coords,
    cycle_edge_list,
    title="",
    view=cycle_styled_view,
    root_node_index=cycle_root_node_index,
    axis_permutation=cycle_styled_axis_permutation,
);


### Skewed Low-Rank Spectral Bias


In [ ]:
# Spectral-bias constants for this block
cycle_spectral_drop_top_eigenvector = True
cycle_spectral_reorder_prefix = None
cycle_spectral_cutoff = None
cycle_spectral_figsize = (15.0, 6.0)
cycle_spectral_legend_anchor = (0.0, 1.3)

cycle_norm_eigenvalues, cycle_norm_projections = compute_spectral_bias_from_state(
    embeddings=cycle_node_embeddings,
    spectral_state=cycle_laplacian_state,
    drop_top_eigenvector=cycle_spectral_drop_top_eigenvector,
    reorder_prefix=cycle_spectral_reorder_prefix,
)

print("Cycle normalized eigenvalues:\n", cycle_norm_eigenvalues.round(3))

plot_spectral_bias(
    norm_eigenvalues=cycle_norm_eigenvalues,
    norm_projections=cycle_norm_projections,
    title="",
    cutoff=cycle_spectral_cutoff,
    figsize=cycle_spectral_figsize,
    ylabel_fontsize=28,
    xlabel_fontsize=28,
    tick_fontsize=20,
    legend_fontsize=20,
    legend_anchor=cycle_spectral_legend_anchor,
);


## 4) Tiny Irregular (N=16, Fixed Edge List)


In [ ]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents) if (candidate / "tiny_graphs_notebooks").is_dir()),
    CURRENT_DIR,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import matplotlib.pyplot as plt

from tiny_graphs_notebooks.notebook_utils.node2vec_notebook_helpers import (
    build_node2vec_section_context,
    train_or_load_node2vec_model,
    evaluate_node2vec_edges,
    print_node2vec_evaluation_report,
    collect_node2vec_embeddings_and_edges,
    load_node2vec_analysis_histories,
)
from tiny_graphs_notebooks.analysis.analysis import (
    build_graph_spectral_state,
    compute_associative_geometric_curves,
    compute_laplacian_coordinates,
    compute_spectral_bias_from_state,
    get_star_branch_layout,
    plot_associative_vs_geometric_curves,
    plot_spectral_bias,
    plot_ordered_heatmaps,
    plot_stylized_embedding_graph,
    reduce_embeddings_for_plot,
    select_embedding_snapshot,
)


In [ ]:
irregular_seed = 7

irregular_train_from_scratch = True
irregular_checkpoint_path = ""
irregular_embedding_history_path = ""
irregular_topk_history_path = ""

irregular_config = {
    "graph_type": "irregular",
    "total_nodes": 16,
    "irregular_edge_count": 20,
    "path_length": 5,

    "embedding_dim": 100,
    "learning_rate": 0.01,
    "num_epochs": 10000,
    "neg_samples_per_pos": 3,
    "use_full_softmax_objective": True,
    "embedding_checkpoint_interval": 25,
    "top_k": 5,
}


In [ ]:
irregular_context = build_node2vec_section_context(irregular_config, seed=irregular_seed)

irregular_resolved_checkpoint_path = train_or_load_node2vec_model(
    irregular_context,
    train_from_scratch=irregular_train_from_scratch,
    checkpoint_path=irregular_checkpoint_path,
)

print("Run name:", irregular_context.run_name)
print("Checkpoint:", irregular_resolved_checkpoint_path)
print("Manifest:", irregular_context.manifest_path)
print("Embedding history pickle:", irregular_context.embedding_history_path)
print("Top-k history pickle:", irregular_context.topk_history_path)


In [ ]:
irregular_eval_metrics = evaluate_node2vec_edges(irregular_context)

print_node2vec_evaluation_report("Tiny Irregular", irregular_eval_metrics)


In [ ]:
irregular_node_embeddings, irregular_edge_list, irregular_root_node_index = collect_node2vec_embeddings_and_edges(irregular_context)

irregular_embedding_history, irregular_topk_recovery_history = load_node2vec_analysis_histories(
    irregular_context,
    embedding_history_path=irregular_embedding_history_path,
    topk_history_path=irregular_topk_history_path,
)

if not irregular_embedding_history:
    irregular_embedding_history = {0: irregular_node_embeddings}
if not irregular_topk_recovery_history:
    irregular_topk_recovery_history = {0: irregular_eval_metrics['topk_edge_recovery_percent']}

print("Node embedding shape:", irregular_node_embeddings.shape)
print("Number of directed edges:", len(irregular_edge_list))
print("Root node index:", irregular_root_node_index)
print("Embedding history steps:", len(irregular_embedding_history))
print("Top-k history steps:", len(irregular_topk_recovery_history))


In [ ]:
# Reduction config for this block (kept local to reduction stage)
irregular_use_umap = False
irregular_reduction_dim = 3
irregular_umap_n_neighbors = 8
irregular_umap_min_dist = 0.3

irregular_reduced_full, irregular_reduced_embeddings = reduce_embeddings_for_plot(
    irregular_node_embeddings,
    use_umap=irregular_use_umap,
    reduction_dim=irregular_reduction_dim,
    seed=irregular_seed,
    umap_n_neighbors=irregular_umap_n_neighbors,
    umap_min_dist=irregular_umap_min_dist,
)

irregular_reduced_embeddings = irregular_reduced_full[:, :3]
print("Reduced full shape:", irregular_reduced_full.shape)
print("Reduced xyz shape:", irregular_reduced_embeddings.shape)


In [ ]:
%matplotlib inline
# %matplotlib widget

# Raw plotting constants for this block
irregular_raw_elev = 10
irregular_raw_azim = 40
irregular_raw_roll = 0
irregular_raw_axis_permutation = (0, 1, 2)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

_xi, _yi, _zi = irregular_raw_axis_permutation
ax.scatter(
    irregular_reduced_embeddings[:, _xi],
    irregular_reduced_embeddings[:, _yi],
    irregular_reduced_embeddings[:, _zi],
    s=180,
    alpha=0.95,
    edgecolor='black',
    linewidth=0.4,
)

for u, v in irregular_edge_list:
    ax.plot(
        [irregular_reduced_embeddings[u, _xi], irregular_reduced_embeddings[v, _xi]],
        [irregular_reduced_embeddings[u, _yi], irregular_reduced_embeddings[v, _yi]],
        [irregular_reduced_embeddings[u, _zi], irregular_reduced_embeddings[v, _zi]],
        color='k',
        linewidth=0.9,
    )

ax.view_init(elev=float(irregular_raw_elev), azim=float(irregular_raw_azim), roll=float(irregular_raw_roll))
plt.title("")
plt.show()


In [ ]:
# Styled plotting constants for this block
irregular_styled_view = {"elev": irregular_raw_elev, "azim": irregular_raw_azim, "roll": irregular_raw_roll}
irregular_styled_axis_permutation = (0, 1, 2)

plot_stylized_embedding_graph(
    irregular_reduced_embeddings,
    irregular_edge_list,
    title="",
    view=irregular_styled_view,
    root_node_index=irregular_root_node_index,
    axis_permutation=irregular_styled_axis_permutation,
);


In [ ]:
# Line-plot constants for this block
irregular_line_plot_title = ""

irregular_curve_steps, irregular_curve_associative, irregular_curve_geometric = compute_associative_geometric_curves(
    irregular_embedding_history,
    irregular_topk_recovery_history,
    irregular_edge_list,
)

plot_associative_vs_geometric_curves(
    steps=irregular_curve_steps,
    associative_scores=irregular_curve_associative,
    geometric_scores=irregular_curve_geometric,
    title=irregular_line_plot_title,
);


Note setting `irregular_heatmap_epoch` to one of the highest geometric_level epochs.


In [ ]:
# Heatmap constants for this block
irregular_heatmap_graph_type = irregular_context.graph_type
irregular_heatmap_epoch = -1
irregular_heatmap_cmap_name = "plasma"
irregular_heatmap_wspace = 0.5

irregular_heatmap_embeddings, irregular_heatmap_resolved_epoch = select_embedding_snapshot(
    embedding_history=irregular_embedding_history,
    fallback_embeddings=irregular_node_embeddings,
    requested_step=irregular_heatmap_epoch,
)
print("Irregular heatmap epoch:", irregular_heatmap_resolved_epoch)
irregular_heatmap_order = list(range(irregular_heatmap_embeddings.shape[0]))
print("Irregular heatmap order:")
print(" ".join(str(node_id) for node_id in irregular_heatmap_order))

plot_ordered_heatmaps(
    embeddings=irregular_heatmap_embeddings,
    edge_list=irregular_edge_list,
    graph_type=irregular_heatmap_graph_type,
    root_node_index=irregular_root_node_index,
    custom_order=irregular_heatmap_order,
    cmap_name=irregular_heatmap_cmap_name,
    wspace=irregular_heatmap_wspace,
);


### Laplacian Geometry Plot


In [ ]:
irregular_styled_view = {"elev": 10, "azim": 40, "roll": 0}

# Laplacian geometry constants for this block
irregular_laplacian_axis_indices = (-2, -3, -4)
irregular_laplacian_state = build_graph_spectral_state(
    edge_list=irregular_edge_list,
    node_count=irregular_node_embeddings.shape[0],
)
irregular_laplacian_coords = compute_laplacian_coordinates(
    spectral_state=irregular_laplacian_state,
    axis_indices=irregular_laplacian_axis_indices,
)

plot_stylized_embedding_graph(
    irregular_laplacian_coords,
    irregular_edge_list,
    title="",
    view=irregular_styled_view,
    root_node_index=irregular_root_node_index,
    axis_permutation=irregular_styled_axis_permutation,
);


### Skewed Low-Rank Spectral Bias


In [ ]:
# Spectral-bias constants for this block
irregular_spectral_drop_top_eigenvector = True
irregular_spectral_reorder_prefix = None
irregular_spectral_cutoff = None
irregular_spectral_figsize = (15.0, 6.0)
irregular_spectral_legend_anchor = (0.0, 1.3)

irregular_norm_eigenvalues, irregular_norm_projections = compute_spectral_bias_from_state(
    embeddings=irregular_node_embeddings,
    spectral_state=irregular_laplacian_state,
    drop_top_eigenvector=irregular_spectral_drop_top_eigenvector,
    reorder_prefix=irregular_spectral_reorder_prefix,
)

print("Irregular normalized eigenvalues:\n", irregular_norm_eigenvalues.round(3))

plot_spectral_bias(
    norm_eigenvalues=irregular_norm_eigenvalues,
    norm_projections=irregular_norm_projections,
    title="",
    cutoff=irregular_spectral_cutoff,
    figsize=irregular_spectral_figsize,
    ylabel_fontsize=28,
    xlabel_fontsize=28,
    tick_fontsize=20,
    legend_fontsize=20,
    legend_anchor=irregular_spectral_legend_anchor,
);
